# RNNs sur des équations mathématiques

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Installation et import de PyTorch Lightning et des autres librairies nécessaires

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import operator

import lightning
import matplotlib.pyplot as plt
import numpy
import sklearn.model_selection
import torch
import torchmetrics
from lightning.pytorch.callbacks import Callback
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Définition du vocabulaire

Dans ces travaux pratiques, nous allons définir des équations, comme `3 + 1 = 4`. Le modèle devra prédire `4` avec comme entrée `3 + 1`.

Pour commencer, créons le vocabulaire :

In [ ]:
operations = list("+*-/")
numbers = list("0123456789.")
padding = [" "]

index_to_char = numbers + operations + padding
char_to_index = {c: i for i, c in enumerate(index_to_char)}

print(f"Index vers caractère : {index_to_char}")
print(f"Caractère vers index : {char_to_index}")

## Utilisation du vocabulaire pour encoder et décoder des équations

Nous pouvons maintenant utiliser ce vocabulaire pour transformer des équations textuelles en suite de chiffres, compréhensibles par un réseau de neurones :

In [ ]:
def encode(characters: str) -> torch.Tensor:
  return torch.tensor([char_to_index[char] for char in characters])


def decode(array: torch.Tensor) -> str:
  return ''.join(index_to_char[int(i)] for i in array)

*Testez les fonctions `encode` et `decode`. Pensez-vous que le réseau de neurones pourra travailler directement sur la sortie de `encode` ou faudra-t-il appliquer un prétraitement supplémentaire ?*

In [ ]:
# Votre code de test ici

Votre réponse ici

### Solution

In [ ]:
equation = "3/4+2-5"
result = "0"
print("─" * 50)
print("Équation")
print("─" * 50)
print(f"Forme brute       {equation}")
print(f"Encodage          {encode(equation)}")
print(f"Encodage/décodage {decode(encode(equation))}")
print()
print("─" * 50)
print("Résultat")
print("─" * 50)
print(f"Forme brute       {result}")
print(f"Encodage          {encode(result)}")
print(f"Encodage/décodage {decode(encode(result))}")

Cet encodage est insuffisant pour le traitement par des réseaux de neurones : il faudra au choix one-hot encoder en sus ou passer par une couche d'embeddings.

## Génération de données

Nous pouvons maintenant procéder à la génération d'exemples, qui serviront pour l'entraînement, la validation et le test.

Pour faire cela, nous allons sélectionner aléatoirement une opération parmi les 4 définies et générer des entiers aléatoires pour appliquer cette opération. Ce processus sera répété jusqu'à atteindre le nombre souhaité d'exemples.

In [ ]:
def make_maths_problem(n_samples: int = 1000,
                       n_digits: int = 3,
                       invert: bool = True
                       ) -> tuple[torch.Tensor, torch.Tensor]:
  equations = []
  results = []
  seen = set()

  math_operation = {"+": operator.add,
                    "-": operator.sub,
                    "*": operator.mul,
                    "/": operator.truediv}

  # Taille maximale que peut faire la chaîne décrivant l'opération
  max_equation_len = 2 * n_digits + 1
  max_result_len = 2 * n_digits

  while len(equations) < n_samples:
    # Sélection d'une opération aléatoire
    operation = numpy.random.choice(operations)

    # Génération de deux entiers qui respectent la limite de taille
    left, right = numpy.random.randint(10 ** n_digits, size=2)

    equation = f"{left}{operation}{right}"

    if equation not in seen:
      seen.add(equation)

      # Calcul du résultat. Étant donné que left et right sont des entiers
      # numpy, ce calcul ne cause pas d'exception, même en cas de division par 0
      math_result = math_operation[operation](left, right)

      # On recommence si le résultat n'est pas exploitable
      if math_result == numpy.inf or numpy.isnan(math_result):
        continue

      # Le résultat peut-être très grand (0.3333333…), on limite sa taille
      result = str(math_result)[:max_result_len]

      # On « pad » pour que toutes les séquences fassent la même taille
      padded_equation = equation.ljust(max_equation_len)
      padded_result = result.ljust(max_result_len)

      # On inverse si l'argument invert est donné
      if invert:
        padded_equation = padded_equation[::-1]

      equations.append(padded_equation)
      results.append(padded_result)

  X = torch.stack(list(map(encode, equations)))
  y = torch.stack(list(map(encode, results)))
  return X, y

Testons cette fonction avec une dizaine d'exemples :

In [ ]:
X, y = make_maths_problem(10, n_digits=3)

print(f"Forme de X : {tuple(X.shape)}")
print(f"Forme de y : {tuple(y.shape)}")

print()
print("Quelques exemples générés")
for encoded_equation, encoded_result in zip(X, y):
  # Par défault l'inversion des équations est activée, il faut la défaire
  # pour pouvoir visualiser l'équation originale
  equation = decode(encoded_equation.flip(0))
  result = decode(encoded_result)
  print(f"{equation} = {result}")

Nous pouvons maintenant créer le dataset et les splits nécessaires :

In [ ]:
X, y = make_maths_problem(100_000, n_digits=3)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, train_size=0.8)

## Définition du modèle

Pour traiter ces séquences de caractères, nous allons utiliser un RNN. Comme dit auparavant, il sera nécessaire de one-hot encoder les séquences d'entrée ou de les passer par une couche d'embeddings. Ici, nous utiliserons la couche d'embeddings.

In [ ]:
class SequencePredictor(lightning.LightningModule):
  """Lightning wrapper: one class is predicted for each output character."""

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-3,
               n_digits: int = 3,
               num_classes: int = len(index_to_char)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, 2 * n_digits + 1,
                                           dtype=torch.long)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val")
    })

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    return self.model(equations)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    equations, results = batch
    # Les logits ont la forme (batch, longueur, vocabulaire), or la perte et la
    # métrique attendent la dimension des classes en deuxième position
    logits = self(equations).transpose(1, 2)
    loss = nn.functional.cross_entropy(logits, results)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, results)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module,
            X: torch.Tensor,
            batch_size: int = 1024) -> torch.Tensor:
  """Apply a model to X, batch by batch, leaving its mode unchanged."""
  device = next(model.parameters()).device
  was_training = model.training
  model.eval()
  logits = torch.cat([model(batch.to(device)).cpu()
                      for batch in X.split(batch_size)])
  model.train(was_training)
  return logits

In [ ]:
class Seq2Seq(nn.Module):
  """Encode the equation, then decode the result character by character."""

  def __init__(self,
               hidden_size: int = 1024,
               n_digits: int = 3,
               embedding_dim: int = 16) -> None:
    super().__init__()
    self.result_len = 2 * n_digits

    # Encodeur : deux couches de GRU empilées, la seconde ne renvoyant que son
    # dernier état caché
    self.embedding = nn.Embedding(len(index_to_char), embedding_dim)
    self.encoder = nn.GRU(embedding_dim,
                          hidden_size,
                          num_layers=2,
                          batch_first=True)

    # Décodeur
    self.decoder = nn.GRU(hidden_size, hidden_size, batch_first=True)

    # La couche de sortie dense est appliquée à chaque pas de temps : nn.Linear
    # ne travaille que sur la dernière dimension du tenseur
    self.head = nn.Linear(hidden_size, len(index_to_char))

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    embedded = self.embedding(equations)
    _, hidden = self.encoder(embedded)

    # Le dernier état caché de l'encodeur résume toute l'équation. On le duplique
    # autant de fois que l'on souhaite de caractères en sortie : c'est un moyen
    # simple de conditionner le décodage du résultat sur l'encodage de l'équation
    summary = hidden[-1].unsqueeze(1).repeat(1, self.result_len, 1)

    decoded, _ = self.decoder(summary)
    return self.head(decoded)


def rnn_model(hidden_size: int = 1024, n_digits: int = 3) -> nn.Module:
  return Seq2Seq(hidden_size, n_digits)


model = SequencePredictor(rnn_model())

## Entraînement du modèle

On affichera régulièrement des prédictions sur des exemples stables pour voir l'évolution du modèle.

In [ ]:
# On utilisera les mêmes exemples à chaque évaluation pour voir le modèle
# progresser
n_test = 20
indexes = torch.randperm(X_test.shape[0])[:n_test]
X_examples, y_examples = X_test[indexes], y_test[indexes]

In [ ]:
def examples(model: nn.Module) -> None:
  guesses = predict(model, X_examples).argmax(dim=-1)
  for encoded_equation, encoded_result, encoded_guess in zip(
      X_examples, y_examples, guesses):
    equation = decode(encoded_equation.flip(0))
    result = decode(encoded_result)
    guess = decode(encoded_guess)
    print(f"{equation} = {result} ?= {guess}")


class PrintExamples(Callback):
  """Show the model's guesses on fixed examples every few epochs."""

  def __init__(self, every_n_epochs: int = 10) -> None:
    super().__init__()
    self.every_n_epochs = every_n_epochs

  def on_validation_epoch_end(self,
                              trainer: lightning.Trainer,
                              pl_module: lightning.LightningModule) -> None:
    epoch = trainer.current_epoch + 1
    if epoch % self.every_n_epochs:
      return
    print()
    print("─" * 50)
    print(f"Après {epoch} epochs :")
    examples(pl_module)


batch_size = 4096
train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=100,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="seq2seq"),
                            enable_checkpointing=False,
                            callbacks=[PrintExamples(every_n_epochs=10)])
trainer.fit(model, train_loader, val_loader)

## Évaluation du modèle

Pour évaluer notre modèle, nous allons calculer 3 éléments sur l'ensemble de test :

- les prédictions illégales (quand la sortie du modèle ne peut pas être interprétée comme un flottant)
- l'accuracy
- l'erreur fractionnelle ($\frac{y_{pred} - y}{y}$)

In [ ]:
def evaluate(model: nn.Module) -> None:
  # Calcul des prédictions du modèle
  predictions = predict(model, X_test).argmax(dim=-1)

  # Décodage des prédictions du modèle en chaîne de caractères
  str_predictions = [decode(prediction) for prediction in predictions]

  # Décodage des prédictions du modèle en flottants
  # Parfois la chaîne de caractères émise ne représente pas un flottant valide
  # On crée un masque booléen pour pouvoir filtrer ces éléments à posteriori
  illegal_predictions_mask = torch.zeros(X_test.shape[0], dtype=torch.bool)
  float_predictions = []
  for i, str_prediction in enumerate(str_predictions):
    try:
      float_prediction = float(str_prediction)
      float_predictions.append(float_prediction)
    except ValueError:
      illegal_predictions_mask[i] = True
  illegal_ratio = int(illegal_predictions_mask.sum()) / X_test.shape[0]
  print(f"Pourcentage de prédictions illégales : {illegal_ratio * 100}%")

  y_pred = torch.tensor(float_predictions)
  y = torch.tensor([float(decode(result)) for result in y_test])
  y = y[~illegal_predictions_mask]

  accuracy = int((y == y_pred).sum()) / X_test.shape[0]
  print(f"Accuracy : {accuracy:.2f}")

  y_denominator = y.clone()
  y_denominator[y == 0] = 1

  fractional_difference = (y_pred - y) / y_denominator

  print(f"Moyenne de l'erreur fractionnelle : {fractional_difference.mean()}")

  plt.hist(fractional_difference.numpy(), bins=1000)
  plt.title("Vue globale de la distribution des erreurs fractionnelles")
  plt.xlabel("Erreur fractionnelle")
  plt.ylabel("Décompte d'exemples")
  plt.xscale("symlog")
  plt.yscale("log")
  plt.show()

  plt.hist(fractional_difference.numpy(), bins=20, range=(-0.1, 0.1))
  plt.title("Vue zoomée sur 0 de la distribution des erreurs fractionnelles")
  plt.xlabel("Erreur fractionnelle")
  plt.ylabel("Décompte d'exemples")
  plt.show()


evaluate(model)